In [1]:
import os
import pandas as pd
import numpy as np
import librosa
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForAudioClassification, AutoFeatureExtractor
from torch.optim import AdamW
from torch.cuda.amp import autocast, GradScaler  
from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report

/media/nit/DATADRIVE1/AJAY/MLproject/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model_id = "facebook/mms-1b-all" 
SAMPLE_RATE = 16000
MAX_DURATION_SECONDS = 2.0
MAX_FRAMES = int(MAX_DURATION_SECONDS * SAMPLE_RATE)

BATCH_SIZE = 4 

EPOCHS = 50  
EARLY_STOPPING_PATIENCE = 8

DATASETS = {
    'train': [
        {'csv': '/media/nit/DATADRIVE1/AJAY/MLproject/LID_MAIN/labeled/labeled_word_timings_train.csv', 'audio': '/media/nit/DATADRIVE1/AJAY/MLproject/ASR/adult_audio/train_split'},
        {'csv': '/media/nit/DATADRIVE1/AJAY/MLproject/LID_MAIN/labeled/children_labeled_word_timings_train.csv', 'audio': '/media/nit/DATADRIVE1/AJAY/MLproject/ASR/audio/train_split'}
    ],
    'val': [
        {'csv': '/media/nit/DATADRIVE1/AJAY/MLproject/LID_MAIN/labeled/labeled_word_timings_val.csv', 'audio': '/media/nit/DATADRIVE1/AJAY/MLproject/ASR/adult_audio/val_split'},
        {'csv': '/media/nit/DATADRIVE1/AJAY/MLproject/LID_MAIN/labeled/children_labeled_word_timings_val.csv', 'audio': '/media/nit/DATADRIVE1/AJAY/MLproject/ASR/audio/val_split'}
    ],
    'test': [
        {'csv': '/media/nit/DATADRIVE1/AJAY/MLproject/LID_MAIN/labeled/labeled_word_timings_test.csv', 'audio': '/media/nit/DATADRIVE1/AJAY/MLproject/ASR/adult_audio/test_split'},
        {'csv': '/media/nit/DATADRIVE1/AJAY/MLproject/LID_MAIN/labeled/children_labeled_word_timings_test.csv', 'audio': '/media/nit/DATADRIVE1/AJAY/MLproject/ASR/audio/test_split'}
    ]
}

SAVE_DIR = "/media/nit/DATADRIVE1/AJAY/MLproject/LID_MAIN/new/mss/"
SAVE_DIR1 = "/media/nit/DATADRIVE1/AJAY/MLproject/LID_MAIN/new/xlsr_finetuned_test"

Using device: cuda:0


In [3]:
print(f"\nLoading {model_id} Model...")
processor = AutoFeatureExtractor.from_pretrained(model_id)

model = AutoModelForAudioClassification.from_pretrained(
    model_id, 
    num_labels=2, 
    use_safetensors=True
)
model.to(device)


Loading facebook/mms-1b-all Model...


2026-04-02 15:43:05.924051: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-02 15:43:05.970976: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-02 15:43:07.516004: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
Some weights of Wav2

Wav2Vec2ForSequenceClassification(
  (wav2vec2): Wav2Vec2Model(
    (feature_extractor): Wav2Vec2FeatureEncoder(
      (conv_layers): ModuleList(
        (0): Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (1-4): 4 x Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (5-6): 2 x Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
      )
    )
    (feature_projection): Wav2Vec2FeatureProjection(
      (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=

In [4]:
class UnifiedLIDDataset(Dataset):
    def __init__(self, split_data_list, processor, max_length):
        self.processor = processor
        self.max_length = max_length
        self.label_map = {"eng": 0, "hin": 1}
        
        dfs = []
        for item in split_data_list:
            df = pd.read_csv(item['csv'])
            df['audio_dir'] = item['audio'] 
            dfs.append(df)
            
        self.df = pd.concat(dfs, ignore_index=True)
        self.df['label'] = self.df['label'].astype(str).str.strip().str.lower()
        
        original_count = len(self.df)
        self.df = self.df[self.df['label'].isin(['eng', 'hin'])].reset_index(drop=True)
        dropped_count = original_count - len(self.df)
        print(f"Loaded {len(self.df)} valid samples. (Deleted {dropped_count} bad/'other' rows)")
        
    def __len__(self):
        return len(self.df)
        
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        file_path = os.path.join(row['audio_dir'], row['segment_id'])
        
        label_str = str(row['label'])
        label = self.label_map[label_str] 
        
        try:
            audio, sr = librosa.load(file_path, sr=SAMPLE_RATE)
            start = int(row['start_time'] * sr)
            end = int(row['end_time'] * sr)
            audio_segment = audio[start:end]
            
            if len(audio_segment) < 300:
                audio_segment = np.zeros(SAMPLE_RATE) 
                label = -100  
                
        except Exception:
            audio_segment = np.zeros(SAMPLE_RATE) 
            label = -100  
            
        inputs = self.processor(
            audio_segment, 
            sampling_rate=SAMPLE_RATE, 
            return_tensors="pt", 
            max_length=self.max_length, 
            truncation=True, 
            padding="max_length"
        )
        
        return inputs.input_values.squeeze(0), torch.tensor(label, dtype=torch.long)

In [5]:
print("\nPreparing Train Data...")
train_dataset = UnifiedLIDDataset(DATASETS['train'], processor, MAX_FRAMES)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

print("Preparing Validation Data...")
val_dataset = UnifiedLIDDataset(DATASETS['val'], processor, MAX_FRAMES)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("Preparing Test Data...")
test_dataset = UnifiedLIDDataset(DATASETS['test'], processor, MAX_FRAMES)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

optimizer = AdamW(model.parameters(), lr=1e-5)


Preparing Train Data...
Loaded 35272 valid samples. (Deleted 16 bad/'other' rows)
Preparing Validation Data...
Loaded 5108 valid samples. (Deleted 1 bad/'other' rows)
Preparing Test Data...
Loaded 9971 valid samples. (Deleted 8 bad/'other' rows)


In [6]:
scaler = GradScaler()

def scan_for_dropped_files(dataset, name="Dataset"):
    print(f"\nScanning {name} for short/corrupted files (-100 flags)...")
    scan_loader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=4)
    dropped_count = 0
    total_files = len(dataset)
    for _, batch_labels in tqdm(scan_loader, desc=f"Scanning {name}"):
        dropped_count += (batch_labels == -100).sum().item()
        
    valid_count = total_files - dropped_count
    print(f"Result for {name}:")
    print(f"  -> Total files scanned: {total_files}")
    print(f"  -> Valid files: {valid_count}")
    print(f"  -> Dropped (-100) files: {dropped_count}")
    return dropped_count

train_dropped = scan_for_dropped_files(train_dataset, "Train Data")
val_dropped = scan_for_dropped_files(val_dataset, "Validation Data")
test_dropped = scan_for_dropped_files(test_dataset, "Test Data")

print(f"\nTotal -100 files across all datasets: {train_dropped + val_dropped + test_dropped}")

/tmp/ipykernel_16567/57819759.py:1: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()



Scanning Train Data for short/corrupted files (-100 flags)...


Scanning Train Data: 100%|█████████████████| 1103/1103 [00:10<00:00, 104.57it/s]


Result for Train Data:
  -> Total files scanned: 35272
  -> Valid files: 35272
  -> Dropped (-100) files: 0

Scanning Validation Data for short/corrupted files (-100 flags)...


Scanning Validation Data: 100%|███████████████| 160/160 [00:03<00:00, 48.60it/s]


Result for Validation Data:
  -> Total files scanned: 5108
  -> Valid files: 5108
  -> Dropped (-100) files: 0

Scanning Test Data for short/corrupted files (-100 flags)...


Scanning Test Data: 100%|█████████████████████| 312/312 [00:04<00:00, 67.65it/s]

Result for Test Data:
  -> Total files scanned: 9971
  -> Valid files: 9971
  -> Dropped (-100) files: 0

Total -100 files across all datasets: 0


In [7]:
print(f"\n--- Loading BEST Model for Final Testing ---")
model = AutoModelForAudioClassification.from_pretrained(SAVE_DIR, use_safetensors=True).to(device)
model.eval()

all_preds = []
all_labels = []
xlsr_probs = [] 
valid_indices = [] 
current_row_idx = 0

test_loop = tqdm(test_loader, desc="Testing")
with torch.no_grad():
    for batch_audio, batch_labels in test_loop:
        batch_audio = batch_audio.to(device)
        outputs = model(batch_audio)
        
        probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()
        predictions = np.argmax(probs, axis=-1)
     
        for i in range(len(batch_labels)):
            if batch_labels[i].item() != -100:
                all_preds.append(predictions[i])
                all_labels.append(batch_labels[i].item())
                xlsr_probs.append(probs[i]) 
                valid_indices.append(current_row_idx)
            current_row_idx += 1

mms_probs = np.array(xlsr_probs)

final_acc = accuracy_score(all_labels, all_preds)
print(f"\n MMS FINAL TEST ACCURACY: {final_acc * 100:.2f}% \n")


--- Loading BEST Model for Final Testing ---


Testing: 100%|██████████████████████████████| 2493/2493 [02:56<00:00, 14.09it/s]


 MMS FINAL TEST ACCURACY: 91.47% 



In [10]:

def extract_pytorch_probabilities(dataloader, dataset_df, mms, output_csv_path):
    print(f"Extracting {mms} Probabilities...")
    model.eval()
    
    all_probs = []
    valid_indices = [] 
    current_row_idx = 0

    with torch.no_grad():
        for batch_audio, batch_labels in tqdm(dataloader, desc="Extracting"):
            batch_audio = batch_audio.to(device)
            
            with autocast(): 
                outputs = model(batch_audio)
                probabilities = torch.softmax(outputs.logits, dim=-1)
                
            hindi_probs = probabilities[:, 1].cpu().numpy()
         
            for i in range(len(batch_labels)):
                if batch_labels[i].item() != -100:
                    all_probs.append(hindi_probs[i])
                    valid_indices.append(current_row_idx)
                current_row_idx += 1

    output_df = dataset_df.iloc[valid_indices][['segment_id', 'start_time', 'end_time', 'label']].copy()
    output_df[f'{mms}_prob_hin'] = all_probs
    
    output_df.to_csv(output_csv_path, index=False)
    print(f"Saved to: {output_csv_path}\n")

# Call this on your Val and Test loaders
extract_pytorch_probabilities(val_loader, val_dataset.df, 'mms', '/media/nit/DATADRIVE1/AJAY/MLproject/LID_MAIN/new/probability/mms_val_probs.csv')
extract_pytorch_probabilities(test_loader, test_dataset.df, 'mms', '/media/nit/DATADRIVE1/AJAY/MLproject/LID_MAIN/new/probability/mms_test_probs.csv')
# Example: extract_pytorch_probabilities(val_loader, val_dataset.df, 'xlsr', 'xlsr_val_probs.csv')

Extracting mms Probabilities...


Extracting:   0%|                                      | 0/1277 [00:00<?, ?it/s]/tmp/ipykernel_16567/509335604.py:13: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Extracting: 100%|███████████████████████████| 1277/1277 [00:44<00:00, 28.47it/s]


Saved to: /media/nit/DATADRIVE1/AJAY/MLproject/LID_MAIN/new/probability/mms_val_probs.csv

Extracting mms Probabilities...


Extracting:   0%|                                      | 0/2493 [00:00<?, ?it/s]/tmp/ipykernel_16567/509335604.py:13: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Extracting: 100%|███████████████████████████| 2493/2493 [01:29<00:00, 27.78it/s]

Saved to: /media/nit/DATADRIVE1/AJAY/MLproject/LID_MAIN/new/probability/mms_test_probs.csv



In [11]:
print("\nLoading XLS-R Model...")
model_id = "facebook/wav2vec2-xls-r-300m" 
processor = AutoFeatureExtractor.from_pretrained(model_id)

model = AutoModelForAudioClassification.from_pretrained(
    model_id, 
    num_labels=2, 
    use_safetensors=True
)
model.to(device)


Loading XLS-R Model...


Some weights of Wav2Vec2ForSequenceClassification were not initialized from the model checkpoint at facebook/wav2vec2-xls-r-300m and are newly initialized: ['classifier.bias', 'classifier.weight', 'projector.bias', 'projector.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Wav2Vec2ForSequenceClassification(
  (wav2vec2): Wav2Vec2Model(
    (feature_extractor): Wav2Vec2FeatureEncoder(
      (conv_layers): ModuleList(
        (0): Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (1-4): 4 x Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (5-6): 2 x Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
      )
    )
    (feature_projection): Wav2Vec2FeatureProjection(
      (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=

In [12]:
print("\nPreparing Train Data...")
train_dataset = UnifiedLIDDataset(DATASETS['train'], processor, MAX_FRAMES)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

print("Preparing Validation Data...")
val_dataset = UnifiedLIDDataset(DATASETS['val'], processor, MAX_FRAMES)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("Preparing Test Data...")
test_dataset = UnifiedLIDDataset(DATASETS['test'], processor, MAX_FRAMES)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

optimizer = AdamW(model.parameters(), lr=1e-5)


Preparing Train Data...
Loaded 35272 valid samples. (Deleted 16 bad/'other' rows)
Preparing Validation Data...
Loaded 5108 valid samples. (Deleted 1 bad/'other' rows)
Preparing Test Data...
Loaded 9971 valid samples. (Deleted 8 bad/'other' rows)


In [13]:

def scan_for_dropped_files(dataset, name="Dataset"):
    print(f"\nScanning {name} for short/corrupted files (-100 flags)...")
    
    scan_loader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=4)
    
    dropped_count = 0
    total_files = len(dataset)
    
    for _, batch_labels in tqdm(scan_loader, desc=f"Scanning {name}"):

        dropped_count += (batch_labels == -100).sum().item()
        
    valid_count = total_files - dropped_count
    print(f"Result for {name}:")
    print(f"  -> Total files scanned: {total_files}")
    print(f"  -> Valid files: {valid_count}")
    print(f"  -> Dropped (-100) files: {dropped_count}")
    return dropped_count


train_dropped = scan_for_dropped_files(train_dataset, "Train Data")
val_dropped = scan_for_dropped_files(val_dataset, "Validation Data")
test_dropped = scan_for_dropped_files(test_dataset, "Test Data")

print(f"\nTotal -100 files across all datasets: {train_dropped + val_dropped + test_dropped}")


Scanning Train Data for short/corrupted files (-100 flags)...


Scanning Train Data: 100%|█████████████████| 1103/1103 [00:09<00:00, 119.45it/s]


Result for Train Data:
  -> Total files scanned: 35272
  -> Valid files: 35272
  -> Dropped (-100) files: 0

Scanning Validation Data for short/corrupted files (-100 flags)...


Scanning Validation Data: 100%|██████████████| 160/160 [00:01<00:00, 113.05it/s]


Result for Validation Data:
  -> Total files scanned: 5108
  -> Valid files: 5108
  -> Dropped (-100) files: 0

Scanning Test Data for short/corrupted files (-100 flags)...


Scanning Test Data: 100%|████████████████████| 312/312 [00:02<00:00, 117.09it/s]

Result for Test Data:
  -> Total files scanned: 9971
  -> Valid files: 9971
  -> Dropped (-100) files: 0

Total -100 files across all datasets: 0


In [14]:
print(f"\n--- Loading BEST Model for Final Testing ---")
model = AutoModelForAudioClassification.from_pretrained(SAVE_DIR1, use_safetensors=True).to(device)
model.eval()

all_preds = []
all_labels = []
xlsr_probs = [] 
valid_indices = [] 
current_row_idx = 0

test_loop = tqdm(test_loader, desc="Testing")
with torch.no_grad():
    for batch_audio, batch_labels in test_loop:
        batch_audio = batch_audio.to(device)
        outputs = model(batch_audio)
        
        probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()
        predictions = np.argmax(probs, axis=-1)
     
        for i in range(len(batch_labels)):
            if batch_labels[i].item() != -100:
                all_preds.append(predictions[i])
                all_labels.append(batch_labels[i].item())
                xlsr_probs.append(probs[i]) 
                valid_indices.append(current_row_idx)
            current_row_idx += 1

xlsr_probs = np.array(xlsr_probs)

final_acc = accuracy_score(all_labels, all_preds)
print(f"\n XLS-R FINAL TEST ACCURACY: {final_acc * 100:.2f}% \n")


--- Loading BEST Model for Final Testing ---


Testing: 100%|██████████████████████████████| 2493/2493 [01:15<00:00, 32.94it/s]


 XLS-R FINAL TEST ACCURACY: 91.67% 



In [16]:

def extract_pytorch_probabilities(dataloader, dataset_df, xlsr, output_csv_path):
    print(f"Extracting {xlsr} Probabilities...")
    model.eval()
    
    all_probs = []
    valid_indices = [] 
    current_row_idx = 0

    with torch.no_grad():
        for batch_audio, batch_labels in tqdm(dataloader, desc="Extracting"):
            batch_audio = batch_audio.to(device)
            
            with autocast(): 
                outputs = model(batch_audio)
                probabilities = torch.softmax(outputs.logits, dim=-1)
                
            hindi_probs = probabilities[:, 1].cpu().numpy()
         
            for i in range(len(batch_labels)):
                if batch_labels[i].item() != -100:
                    all_probs.append(hindi_probs[i])
                    valid_indices.append(current_row_idx)
                current_row_idx += 1

    output_df = dataset_df.iloc[valid_indices][['segment_id', 'start_time', 'end_time', 'label']].copy()
    output_df[f'{xlsr}_prob_hin'] = all_probs
    
    output_df.to_csv(output_csv_path, index=False)
    print(f"Saved to: {output_csv_path}\n")

# Call this on your Val and Test loaders
extract_pytorch_probabilities(val_loader, val_dataset.df, 'xlsr', '/media/nit/DATADRIVE1/AJAY/MLproject/LID_MAIN/new/probability/xlsr_val_probs.csv')
extract_pytorch_probabilities(test_loader, test_dataset.df, 'xlsr', '/media/nit/DATADRIVE1/AJAY/MLproject/LID_MAIN/new/probability/xlsr_test_probs.csv')
# Example: extract_pytorch_probabilities(val_loader, val_dataset.df, 'xlsr', 'xlsr_val_probs.csv')

Extracting xlsr Probabilities...


Extracting:   0%|                                      | 0/1277 [00:00<?, ?it/s]/tmp/ipykernel_16567/265318260.py:13: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Extracting: 100%|███████████████████████████| 1277/1277 [00:22<00:00, 56.92it/s]


Saved to: /media/nit/DATADRIVE1/AJAY/MLproject/LID_MAIN/new/probability/xlsr_val_probs.csv

Extracting xlsr Probabilities...


Extracting:   0%|                                      | 0/2493 [00:00<?, ?it/s]/tmp/ipykernel_16567/265318260.py:13: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Extracting: 100%|███████████████████████████| 2493/2493 [00:44<00:00, 56.20it/s]


Saved to: /media/nit/DATADRIVE1/AJAY/MLproject/LID_MAIN/new/probability/xlsr_test_probs.csv

